<a href="https://colab.research.google.com/github/hafizihsani/data-science-2026/blob/main/Pertemuan3_Muhamad_Hafiz_ihsani_250401020155.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
# 1 & 2. Import Libraries
import pandas as pd
import numpy as np
from scipy.stats.mstats import winsorize
import requests

In [18]:
# 3. Muat Dataset
# Menggunakan URL Drive yang diberikan
url_path = 'https://drive.google.com/uc?id=1LfQWProB0VjWN5q8bKuRIgn-stULfIRo'
df = pd.read_csv(url_path)

In [19]:
# 4. Eksplorasi Awal (Ringkas dalam satu blok)
print("--- Informasi Dataset ---")
df.info()
print("\n--- Statistik Deskriptif ---")
display(df.describe())
print("\n--- Jumlah Missing Values ---")
print(df.isnull().sum())

--- Informasi Dataset ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 130 entries, 0 to 129
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            130 non-null    int64  
 1   luas_m2       112 non-null    float64
 2   harga_juta    113 non-null    float64
 3   kota          130 non-null    object 
 4   kamar         120 non-null    float64
 5   tahun_bangun  130 non-null    int64  
 6   kondisi       130 non-null    object 
dtypes: float64(3), int64(2), object(2)
memory usage: 7.2+ KB

--- Statistik Deskriptif ---


,id,luas_m2,harga_juta,kamar,tahun_bangun
count,130.000000,112.000000,1.130000e+02,120.000000,130.000000
mean,65.500000,267.627679,8.856325e+05,3.433333,2062.638462
std,37.671829,885.664181,9.407144e+06,1.776283,701.684043
min,1.000000,-50.000000,-5.000000e+02,1.000000,1890.000000
25%,33.250000,87.050000,3.450000e+02,2.000000,1991.250000
50%,65.500000,193.800000,6.550000e+02,4.000000,2002.000000
75%,97.750000,280.675000,9.550000e+02,5.000000,2011.750000
max,130.000000,9500.000000,1.000000e+08,6.000000,9999.000000



--- Jumlah Missing Values ---
id               0
luas_m2         18
harga_juta      17
kota             0
kamar           10
tahun_bangun     0
kondisi          0
dtype: int64


In [21]:
# 5. Penanganan Duplikat
count_dup = df.duplicated().sum()
if count_dup > 0:
    df.drop_duplicates(keep='first', inplace=True)
    print(f"\nBerhasil menghapus {count_dup} baris duplikat.")

In [24]:
# 6. Normalisasi String (Menggunakan metode apply atau mapping untuk variasi)
# Normalisasi kolom 'kondisi' menjadi lowercase dan tanpa spasi
df['kondisi'] = df['kondisi'].apply(lambda x: str(x).strip().lower() if pd.notnull(x) else x)

In [25]:
# Normalisasi kolom 'kota' menjadi Title Case
df['kota'] = df['kota'].str.strip().str.title()
print("\nNormalisasi string selesai (Kondisi -> lower, Kota -> Title).")


Normalisasi string selesai (Kondisi -> lower, Kota -> Title).


In [28]:
# 7. Imputasi Missing Values
# Memisahkan kolom berdasarkan tipe data untuk imputasi otomatis
numerik_cols = ['luas_m2', 'harga_juta']
kategorik_cols = ['kamar']

# Imputasi Median untuk Numerik
for col in numerik_cols:
    df[col] = df[col].replace(np.nan, df[col].median())

# Imputasi Modus untuk Kategorik/Diskrit
for col in kategorik_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

In [29]:
# 8. Penanganan Outlier dengan IQR Fence
# Fungsi kustom untuk clipping berdasarkan IQR
def handle_outliers(dataframe, columns):
    for col in columns:
        Q1 = dataframe[col].quantile(0.25)
        Q3 = dataframe[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        # Menggunakan np.clip untuk efisiensi
        dataframe[col] = np.clip(dataframe[col], lower_bound, upper_bound)
    return dataframe

df = handle_outliers(df, ['harga_juta', 'luas_m2'])

In [30]:
# 9. Validasi Akhir
check_null = df.isnull().sum().sum()
check_dup = df.duplicated().sum()

print("\n--- Hasil Validasi ---")
print(f"Total Missing Values: {check_null}")
print(f"Total Duplikat      : {check_dup}")

if check_null == 0 and check_dup == 0:
    print("Validasi Berhasil: Data sudah bersih.")
else:
    print("Peringatan: Masih ditemukan masalah pada data.")

# 10. Ekspor Data
df.to_csv('housing_clean.csv', index=False)
print("\nFile 'housing_clean.csv' telah disimpan.")

# 11. Akses API JSONPlaceholder
print("\n--- Mengambil Data dari API ---")
api_url = "https://jsonplaceholder.typicode.com/posts"
res = requests.get(api_url)

if res.status_code == 200:
    df_api = pd.DataFrame(res.json())
    print("Berhasil mengambil data dari API.")
    display(df_api.head())
else:
    print(f"Gagal akses API. Status code: {res.status_code}")

# Akhir dari kode


--- Hasil Validasi ---
Total Missing Values: 0
Total Duplikat      : 0
Validasi Berhasil: Data sudah bersih.

File 'housing_clean.csv' telah disimpan.

--- Mengambil Data dari API ---
Berhasil mengambil data dari API.


,userId,id,title,body
0,1,1,sunt aut facere repellat provident occaecati e...,quia et suscipit\nsuscipit recusandae consequu...
1,1,2,qui est esse,est rerum tempore vitae\nsequi sint nihil repr...
2,1,3,ea molestias quasi exercitationem repellat qui...,et iusto sed quo iure\nvoluptatem occaecati om...
3,1,4,eum et est occaecati,ullam et saepe reiciendis voluptatem adipisci\...
4,1,5,nesciunt quas odio,repudiandae veniam quaerat sunt sed\nalias aut...
